# 66 Protected-point preservation check

Thesis §3.5 defines the preservation invariant as the Euclidean distance from each protected point to the nearest surviving vertex of the anonymized mesh. This notebook computes that metric for every valid subject and also reports the same distance against the *original* mesh, so that picker-precision effects (landmarks clicked in sparse regions of the triangulation) can be separated from pipeline displacement.

Both distances come from the shipped `check_protected_points` function in `cedalion.geometry.photogrammetry.anonymization`. The thesis numbers are produced by the function the codebase exports, not a notebook re-implementation of the KDTree query.

- `d_anon_mm` = distance from landmark to nearest vertex of the anonymized mesh.
- `d_orig_mm` = distance from landmark to nearest vertex of the original mesh.
- `delta_mm  = d_anon_mm - d_orig_mm`.

The pipeline claim is $\Delta = 0$: no surviving vertex is repositioned, so the nearest-vertex identity is preserved bit-exact. The absolute `d_anon_mm` characterizes picker precision, not pipeline error, and can exceed the 1 mm edge-length heuristic on sparse regions of the Einstar mesh (e.g. pre-auricular notch, high vertex).

**Cohort.** Iterates over the eleven valid thesis subjects: the optode-cap cohort S1--S7 (Subject 16--22) and the bare-cap cohort S8--S11 (Subject 12--15). Subject 11 is excluded (unusable raw scan). The output CSVs carry the shared `s_id` and `optode` columns so the LaTeX tables can render cohort membership directly.

Populates §4.4.1 of the thesis. Outputs (under `thesis_results_out/`):
- `preservation_check.csv` -- one row per (subject, landmark)
- `preservation_summary.csv` -- one row per subject

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))
from _thesis_data import (
    subject_paths, load_raw, load_anon, load_landmarks,
    available_subjects, is_optode, s_id,
)
from cedalion.geometry.photogrammetry.anonymization import (
    check_protected_points,
)

import pandas as pd

OUT_DIR = pathlib.Path('thesis_results_out')
OUT_DIR.mkdir(exist_ok=True)

ready_with_anon = [
    n for n in available_subjects() if subject_paths(n).anon_exists
]
print(f'ready_with_anon ({len(ready_with_anon)}): {ready_with_anon}')

## 1. Per-landmark table

For each subject, run `check_protected_points` against both the original and anonymized mesh, and unroll the per-landmark deltas it returns into a row per (subject, landmark).

In [ ]:
rows = []
for n in ready_with_anon:
    surf_orig = load_raw(n)
    surf_anon = load_anon(n)
    lm = load_landmarks(n)

    result = check_protected_points(surf_orig, surf_anon, lm)
    for entry in result.per_point:
        rows.append({
            's_id': s_id(n),
            'subject': n,
            'optode': is_optode(n),
            'landmark': entry.label,
            'd_orig_mm': entry.d_orig_mm,
            'd_anon_mm': entry.d_anon_mm,
            'delta_mm': entry.delta_mm,
        })
df = pd.DataFrame(rows)
if len(df):
    df = df.iloc[df['s_id'].map(lambda s: int(s[1:])).argsort(kind='stable')].reset_index(drop=True)
df.head(15)

## 2. Per-subject summary

Max across the five landmarks of `d_anon_mm` (absolute) and `delta_mm` (pipeline displacement). The primary thesis claim is that `max_delta_mm` is 0 bit-exact; `max_d_anon_mm` characterizes edge length at the landmark sites.

In [ ]:
summary = (
    df.groupby(['s_id', 'subject', 'optode'], as_index=False)
    .agg(
        max_d_anon_mm=('d_anon_mm', 'max'),
        max_d_orig_mm=('d_orig_mm', 'max'),
        max_abs_delta_mm=('delta_mm', lambda s: float(s.abs().max())),
    )
)
summary['pass_1mm'] = summary.max_d_anon_mm <= 1.0
summary['pass_2mm'] = summary.max_d_anon_mm <= 2.0
summary = summary.iloc[summary['s_id'].map(lambda s: int(s[1:])).argsort()].reset_index(drop=True)
summary

## 3. Save

In [ ]:
per_landmark_out = OUT_DIR / 'preservation_check.csv'
summary_out = OUT_DIR / 'preservation_summary.csv'
df.to_csv(per_landmark_out, index=False)
summary.to_csv(summary_out, index=False)
print(f'Max |delta| across all subjects and landmarks: {df.delta_mm.abs().max():.3e} mm')
print(f'Max d_anon across all subjects: {summary.max_d_anon_mm.max():.4g} mm')
print(f'Subjects passing 1 mm tolerance: {int(summary.pass_1mm.sum())}/{len(summary)}')
print(f'Subjects passing 2 mm tolerance: {int(summary.pass_2mm.sum())}/{len(summary)}')
print(f'Wrote {per_landmark_out}')
print(f'Wrote {summary_out}')